# DenseNet


## 0. DenseNet 杂项

### 0.1 论文信息

DenseNet 来自论文 `Densely Connected Convolutional Networks`，核心思想是把同一个 dense block 内的每一层都和后续所有层直接连接起来。

| 项目 | 内容 |
| --- | --- |
| 网络名称 | Dense Convolutional Network，简称 DenseNet |
| 主要作者 | Gao Huang, Zhuang Liu, Laurens van der Maaten, Kilian Q. Weinberger |
| 主要任务 | 图像分类 |
| 实验数据集 | CIFAR-10、CIFAR-100、SVHN、ImageNet |
| 核心关键词 | dense connectivity、feature reuse、growth rate、transition layer、bottleneck、compression |

### 0.2 主要解决的问题

普通 CNN 加深后容易出现两个问题：

- 梯度从后往前传播时路径太长，训练困难。
- 前面层学到的低层特征容易在层层传递中被弱化，后面层难以直接利用。

ResNet 用残差连接缓解这个问题，DenseNet 则进一步让每一层都直接读取前面所有层的特征。

### 0.3 一句话理解

DenseNet 不是只把上一层输出传给下一层，而是把当前层之前所有层的特征都拼接起来，再作为当前层的输入。这样每一层都可以复用已有特征，只需要补充少量新特征。


---


## 1. 稠密连接 Dense Connectivity

### 1.1 定义

稠密连接指在一个 dense block 内，第 $l$ 层会接收第 $0$ 层到第 $l-1$ 层的全部特征图作为输入。

普通前馈 CNN 的连接方式是：

$$
x_l = H_l(x_{l-1})
$$

ResNet 的连接方式是：

$$
x_l = H_l(x_{l-1}) + x_{l-1}
$$

DenseNet 的连接方式是：

$$
x_l = H_l([x_0, x_1, \dots, x_{l-1}])
$$

其中：

- $x_l$：第 $l$ 层的输出特征图
- $H_l(\cdot)$：第 $l$ 层的非线性变换，一般由 BN、ReLU、Conv 等操作组成
- $[x_0, x_1, \dots, x_{l-1}]$：把前面所有层的特征图在通道维度上拼接起来

### 1.2 连接数量

如果网络有 $L$ 层，传统 CNN 大约只有相邻层之间的 $L$ 条连接；DenseNet 在 dense block 内会形成大量跨层连接：

$$
\frac{L(L+1)}{2}
$$

这表示网络中存在很多短路径，梯度和特征可以更直接地传播。

### 1.3 直观理解

- 普通 CNN 像接力跑：每层只把结果交给下一层。
- ResNet 像在接力时保留一条旁路：当前变换结果和原输入相加。
- DenseNet 像共享资料库：前面所有层的特征都保留下来，后面层可以按需读取。

### 1.4 注意点

DenseNet 用的是拼接 concat，不是相加 add。拼接会让通道数不断增加，所以后面需要 growth rate、bottleneck 和 transition layer 来控制参数量和计算量。


---


## 2. Dense Block

### 2.1 定义

Dense block 是 DenseNet 的核心模块。一个 dense block 内部包含多层卷积，每一层都会读取该 block 内前面所有层的输出。

### 2.2 基本结构

一个常见的 DenseNet 层可以写成：

$$
H_l(\cdot) = \text{BN} \rightarrow \text{ReLU} \rightarrow \text{Conv}
$$

在 DenseNet-B 中，常见结构变为：

$$
\text{BN} \rightarrow \text{ReLU} \rightarrow 1 \times 1 \text{ Conv} \rightarrow \text{BN} \rightarrow \text{ReLU} \rightarrow 3 \times 3 \text{ Conv}
$$

其中 $1 \times 1$ 卷积用于压缩输入通道，$3 \times 3$ 卷积用于提取局部空间特征。

### 2.3 特征图变化

假设进入 dense block 的通道数是 $c_0$，每层新增 $k$ 个通道，经过 $l$ 层后输入到下一层的通道数大约是：

$$
c_l = c_0 + k \times l
$$

其中：

- $c_0$：进入 dense block 前已有的通道数
- $k$：growth rate，每层新产生的通道数
- $l$：dense block 内已经经过的层数

### 2.4 作用

- 增强特征复用：浅层边缘、纹理等特征可以被深层直接使用。
- 改善梯度传播：损失函数到早期层之间有更短的反向传播路径。
- 减少重复学习：后面层不必重新学习前面已经提取好的特征。

### 2.5 注意点

Dense block 内通常保持特征图的空间尺寸不变，否则不同层输出无法直接拼接。空间尺寸变化一般放在 dense block 之间的 transition layer 中完成。


---


## 3. 增长率 Growth Rate

### 3.1 定义

growth rate 记作 $k$，表示 dense block 中每一层新产生多少个特征图。

### 3.2 为什么需要 growth rate

因为 DenseNet 会把前面所有层的输出都拼接起来，如果每层都输出很多通道，通道数会增长得很快。growth rate 用来控制每层只新增少量特征。

### 3.3 直观理解

DenseNet 把所有已有特征看成一个共享状态。每一层不需要重写全部状态，只需要往里面补充 $k$ 个新的特征图。

| $k$ 的大小 | 影响 |
| --- | --- |
| 较小 | 参数更少，模型更紧凑，但表达能力可能受限 |
| 较大 | 表达能力更强，但通道数、参数量和显存占用会增加 |

### 3.4 论文中的设置

在 CIFAR 和 SVHN 实验中，论文比较了不同深度和不同 $k$ 的 DenseNet；在 ImageNet 的 DenseNet-121、169、201、264 中，growth rate 使用 $k=32$。


---


## 4. 过渡层 Transition Layer

### 4.1 定义

transition layer 位于两个 dense block 之间，用来改变特征图尺寸，并控制通道数。

### 4.2 常见结构

论文中常用的 transition layer 是：

$$
\text{BN} \rightarrow \text{ReLU} \rightarrow 1 \times 1 \text{ Conv} \rightarrow 2 \times 2 \text{ Average Pooling}
$$

其中：

- $1 \times 1$ 卷积：调整通道数。
- $2 \times 2$ average pooling：把空间尺寸下采样，一般让宽高减半。

### 4.3 为什么不用 dense block 内部下采样

Dense block 内部需要拼接来自不同层的特征图。如果中间层随意改变空间尺寸，特征图大小不一致，就无法直接 concat。因此 DenseNet 通常在 block 内保持尺寸不变，在 block 之间统一下采样。

### 4.4 作用

- 降低空间分辨率，扩大后续层的感受野。
- 控制通道数，避免拼接后通道无限膨胀。
- 把多个 dense block 组织成完整 CNN 架构。


---


## 5. Bottleneck 和 Compression

### 5.1 Bottleneck Layer

DenseNet 中每一层虽然只输出 $k$ 个新特征图，但它的输入会包含前面所有层的输出，输入通道可能很多。bottleneck layer 用 $1 \times 1$ 卷积先减少计算量，再做 $3 \times 3$ 卷积。

DenseNet-B 中的一个层通常是：

$$
\text{BN} \rightarrow \text{ReLU} \rightarrow 1 \times 1 \text{ Conv} \rightarrow \text{BN} \rightarrow \text{ReLU} \rightarrow 3 \times 3 \text{ Conv}
$$

论文中常让 $1 \times 1$ 卷积输出 $4k$ 个特征图。

### 5.2 Compression

compression 用于 transition layer。如果输入 transition layer 的通道数是 $m$，压缩系数是 $\theta$，则输出通道数为：

$$
\lfloor \theta m \rfloor
$$

其中：

- $m$：transition layer 输入通道数
- $\theta$：compression factor，满足 $0 < \theta \le 1$
- $\theta=1$：不压缩通道
- $\theta<1$：减少通道数

论文实验中常用 $\theta=0.5$。

### 5.3 DenseNet 变体

| 名称 | 含义 |
| --- | --- |
| DenseNet | 基础稠密连接网络 |
| DenseNet-B | 使用 bottleneck layer |
| DenseNet-C | transition layer 中使用 compression |
| DenseNet-BC | 同时使用 bottleneck 和 compression |

### 5.4 直观理解

DenseNet 的 dense connection 让特征可以充分复用，但会带来通道数增长。bottleneck 和 compression 的作用就是把模型做得更紧凑，减少无效计算和参数浪费。


---


## 6. DenseNet 整体结构

### 6.1 基本流程

一个典型 DenseNet 可以按下面流程理解：

1. 输入图像先经过初始卷积层。
2. 进入第一个 dense block，在 block 内不断拼接和复用特征。
3. 经过 transition layer，下采样并调整通道数。
4. 重复 dense block 和 transition layer。
5. 最后经过 global average pooling 和全连接分类层输出类别概率。

### 6.2 ImageNet 版本

论文给出了 DenseNet-121、DenseNet-169、DenseNet-201、DenseNet-264 等结构。它们的主要区别是各个 dense block 中层数不同。

| 模型 | Dense block 层数配置 | growth rate |
| --- | --- | --- |
| DenseNet-121 | 6, 12, 24, 16 | 32 |
| DenseNet-169 | 6, 12, 32, 32 | 32 |
| DenseNet-201 | 6, 12, 48, 32 | 32 |
| DenseNet-264 | 6, 12, 64, 48 | 32 |

### 6.3 和普通 CNN 的区别

| 结构 | 特征传递方式 | 主要特点 |
| --- | --- | --- |
| 普通 CNN | 当前层只接收上一层输出 | 简单，但深层网络训练困难 |
| ResNet | 当前层输出和残差分支相加 | 梯度更容易传播 |
| DenseNet | 当前层接收前面所有层输出的拼接 | 特征复用更充分，参数效率高 |

### 6.4 分类层

DenseNet 的最后通常使用 global average pooling 汇总空间信息，再接 fully-connected layer 和 softmax 做分类。这样比直接堆很多全连接层更省参数。


---


## 7. 训练和实验 Training and Experiments

### 7.1 实验数据集

| 数据集 | 任务特点 |
| --- | --- |
| CIFAR-10 | 10 类小尺寸彩色图像分类 |
| CIFAR-100 | 100 类小尺寸彩色图像分类，比 CIFAR-10 更难 |
| SVHN | 街景门牌数字识别 |
| ImageNet | 大规模 1000 类图像分类 |

### 7.2 训练设置

论文中主要使用 SGD 训练 DenseNet。在 CIFAR 和 SVHN 上，会根据任务设置 batch size、epoch、学习率衰减、weight decay 和 dropout 等超参数。

需要注意的是，DenseNet 结构本身主要改善的是信息流和特征复用；实际效果仍然依赖数据增强、学习率设置、正则化等训练细节。

### 7.3 主要实验结论

- 在 CIFAR 和 SVHN 上，DenseNet 往往能用较少参数取得较低错误率。
- DenseNet-BC 通常参数效率最好，因为它同时使用 bottleneck 和 compression。
- 在 ImageNet 上，DenseNet 可以用更少参数和计算量达到接近或优于 ResNet 的效果。

### 7.4 参数效率

论文中一个重要结论是：DenseNet 不只是深，而是更会复用特征。因此在相近精度下，它通常比直接加宽或加深的网络更省参数。


---


## 8. DenseNet 和 ResNet 对比

### 8.1 共同点

DenseNet 和 ResNet 都是为了解决深层 CNN 难训练的问题，都通过跨层连接缩短信息和梯度传播路径。

### 8.2 核心区别

| 对比点 | ResNet | DenseNet |
| --- | --- | --- |
| 跨层连接方式 | identity shortcut | dense connection |
| 特征融合方式 | 相加 add | 拼接 concat |
| 当前层输入 | 上一层输出加残差 | 前面所有层输出的拼接 |
| 通道变化 | 一般不会因相加增加通道 | 会随层数增加通道 |
| 主要优势 | 优化更稳定，适合很深网络 | 特征复用强，参数效率高 |

### 8.3 公式对比

ResNet：

$$
x_l = H_l(x_{l-1}) + x_{l-1}
$$

DenseNet：

$$
x_l = H_l([x_0, x_1, \dots, x_{l-1}])
$$

### 8.4 直观理解

ResNet 更像是在学习“相对上一层要改多少”，DenseNet 更像是在不断积累一个特征集合。DenseNet 后面的层可以直接看到前面所有层的结果，因此更容易复用低层和中层特征。

### 8.5 注意点

DenseNet 的 concat 会增加显存占用，尤其是训练时需要保存大量中间特征。实际使用时要注意 batch size、输入分辨率和模型规模。


---


## 9. 特征复用 Feature Reuse

### 9.1 定义

feature reuse 指后面的层可以直接使用前面层已经学到的特征，而不是重新学习相似特征。

### 9.2 为什么 DenseNet 有特征复用

Dense block 内每一层的输出都会被传给后面所有层。浅层的边缘、纹理，中层的局部结构，深层的语义特征，都可以作为后续层的输入。

### 9.3 好处

- 参数更省：不需要每一层重复学习同类特征。
- 梯度更稳：后面层的梯度可以更直接地传回前面层。
- 有正则化效果：小数据集上能一定程度缓解过拟合。
- 模型更紧凑：较窄的层也能利用前面累计的丰富特征。

### 9.4 论文中的观察

论文通过分析训练后卷积层权重发现，DenseNet 的深层确实会使用来自较早层的特征，transition layer 和分类层也会利用 dense block 中多个层的输出。这说明 dense connection 不是形式上的连接，而是实际参与了特征复用。

### 9.5 容易混淆的点

| 概念 | 说明 |
| --- | --- |
| dense connection | 层与层之间密集连接 |
| dense block | 使用 dense connection 的模块 |
| growth rate | 每层新增的特征图数量 |
| bottleneck | 用 $1 \times 1$ 卷积降低计算量 |
| compression | 在 transition layer 中减少通道数 |


---


## 10. DenseNet 总结

### 10.1 核心思想

DenseNet 的核心是：在一个 dense block 内，每一层都接收前面所有层的输出，并把自己的输出传给后面所有层。

### 10.2 主要优点

- 缓解梯度消失，深层网络更容易训练。
- 加强特征传播，早期特征能直接到达后面层。
- 鼓励特征复用，减少重复学习。
- 在相近效果下，参数量通常更少。

### 10.3 主要限制

- concat 会让通道数不断增加，需要控制 growth rate。
- 训练时保存的中间特征较多，显存压力可能比较大。
- 结构实现比普通 CNN 更复杂，需要注意各层特征图尺寸是否一致。

### 10.4 复习重点

学习 DenseNet 时重点掌握四件事：

1. DenseNet 为什么用 concat，而不是像 ResNet 一样 add。
2. dense block 内部为什么要保持空间尺寸一致。
3. growth rate、bottleneck、compression 分别控制什么。
4. DenseNet 为什么能做到较好的特征复用和参数效率。


***********
